In [29]:
import os
import contextlib
import io
import pandas as pd

# from bias_bench.test.automated_test import test_crows

directory = "data/crows_improved/"

to_test_types = ["gender", "race-color", "religion"]
# to_test_languages = ["de_DE", "mt_MT"]
to_test_languages = ["ar_DZ", "ca_ES", "de_DE", "en_US", "es_AR", "fr_FR", "mt_MT", "zh_CN"]

languages = {}

for filename in os.listdir(directory):
    filepath = os.path.join(directory, filename)
    if os.path.isfile(filepath):
        languages[filename[-9:-4]] = directory + filename
        
languages

{'es_AR': 'data/crows_improved/crows_es_AR.csv',
 'ar_DZ': 'data/crows_improved/crows_ar_DZ.csv',
 'fr_FR': 'data/crows_improved/crows_fr_FR.csv',
 'zh_CN': 'data/crows_improved/crows_zh_CN.csv',
 'ca_ES': 'data/crows_improved/crows_ca_ES.csv',
 'de_DE': 'data/crows_improved/crows_de_DE.csv',
 'en_US': 'data/crows_improved/crows_en_US.csv',
 'mt_MT': 'data/crows_improved/crows_mt_MT.csv'}

In [30]:
import difflib
from transformers import AutoTokenizer

def low_strip(word):
    return word.lower().strip(",.")

def extract_changed_words(
    row
):
    sent1 = row["sent_more"]
    sent2 = row["sent_less"]
    
    if pd.isna(sent1) or pd.isna(sent2):
        return None
    
    tokens1 = sent1.split()
    tokens2 = sent2.split()

    matcher = difflib.SequenceMatcher(None, tokens1, tokens2)

    changed_pairs = []

    for tag, i1, i2, j1, j2 in matcher.get_opcodes():
        if tag == 'replace':
            words1 = tokens1[i1:i2]
            words2 = tokens2[j1:j2]
            if len(words1) > 1 and len(words2) > 1:
                for word1, word2 in zip(words1, words2):
                    changed_pairs.append(tuple([
                        (word1),
                        (word2)
                    ]))
            else:
                changed_pairs.append(tuple([
                    (words1[0]),
                    (words2[0])
                ]))

    return [(row.name, w1, w2) for (w1, w2) in changed_pairs]

In [42]:
from collections import Counter
import pandas as pd

def filter_tuples_by_count_with_id(
    tuple_list,
    min_occurrences=2,
    ignore_order=True,
    case_sensitive=False
):
    def norm_key(w1, w2):
        if not case_sensitive:
            w1, w2 = w1.lower(), w2.lower()
        if ignore_order:
            return tuple(sorted([w1, w2]))
        return (w1, w2)

    keys = [norm_key(w1, w2) for _, w1, w2 in tuple_list]

    key_counts = Counter(keys)

    frequent_keys = {k for k, c in key_counts.items() if c > min_occurrences}

    key_to_tuple = {}
    for tup in tuple_list:
        _, w1, w2 = tup
        k = norm_key(w1, w2)
        if k in frequent_keys and k not in key_to_tuple:
            key_to_tuple[k] = tup

    return list(key_to_tuple.values())

In [32]:
import spacy

def select_nlp(lang):
    supported_langs = ["ca", "zh", "hr", "da", "nl", "en", "fi", "fr", "de", "el", "it", "ja", "ko", "lt", "mk", "nb", "pl", "pt", "ro", "ru", "sl", "es", "sv", "uk"]
    lang_abr = lang[0:2]
    if lang_abr not in supported_langs:
        return None
    else:
        if lang_abr in ["zh", "en"]:
            nlp = spacy.load(f"{lang_abr}_core_web_sm")
        else:
            nlp = spacy.load(f"{lang_abr}_core_news_sm")
            
    return nlp

def filter_out_ner(lang, word_pairs, odf, remove_ner={"PERSON", "NORP", "GPE", "LOC"}):
    nlp = select_nlp(lang)
    if nlp is None:
        return word_pairs

    filtered = []

    for eid, word1, word2 in word_pairs:
        doc1 = nlp(word1)
        doc2 = nlp(word2)

        is_ner1 = doc1[0].ent_type_ in remove_ner
        is_ner2 = doc2[0].ent_type_ in remove_ner

        if not (is_ner1 or is_ner2):
            filtered.append((eid, word1, word2))

    print("NER:", len(word_pairs) - len(filtered))

    return filtered

def filter_by_pos(
    lang,
    word_pairs,
    odf,
    keep_pos={'NOUN', 'VERB', 'ADJ', 'ADV', 'PRON'}
):
    nlp = select_nlp(lang)
    if nlp is None:
        return word_pairs
    filtered = []

    for eid, word1, word2 in word_pairs:
        doc1 = nlp(word1)
        doc2 = nlp(word2)

        if len(doc1) == 1 and len(doc2) == 1:
            pos1 = doc1[0].pos_
            pos2 = doc2[0].pos_

            if pos1 in keep_pos and pos2 in keep_pos:
                filtered.append((eid, word1, word2))
                
    print("POS", len(word_pairs) - len(filtered))

    return filtered

In [35]:
from collections import Counter
import re

def filter_by_frequency(extracted_pairs, min_count=2):
    pair_counter = Counter()
    
    for pair in extracted_pairs:
        canonical = tuple(sorted([
            pair[0].lower(),
            pair[1].lower()
        ]))
        pair_counter[canonical] += 1
    
        frequent_pairs = []
        for (w1, w2), count in pair_counter.items():
            if count >= min_count:
                if len(w1) > 1 and len(w2) > 1:
                    pass
                frequent_pairs.append((w1, w2))
    
    
    return frequent_pairs

In [ ]:
import json

attributes = {}

for lang in languages:
    with open(f"data/bias_attribute/bias_attribute_{lang}.json", "w") as f:
        print(lang)
        attributes[lang] = {}
        for btype in to_test_types:
            df = pd.read_csv(languages[lang])
            df = df[df["bias_type"] == btype]
            res = df.apply(extract_changed_words, axis=1)
            
            res = res.dropna()
            res = res[res.str.len() > 0]
            
            big_list = [sublist for item in res for sublist in item]
            
            big_list = filter_tuples_by_count_with_id(big_list)
            
            print(len(big_list), big_list)
                    
            big_list = filter_by_pos(lang, big_list, df)
            
            big_list = filter_out_ner(lang, big_list, df)
            
            print(len(big_list), big_list)
            
            attributes[lang][btype] = big_list
            
            to_save =  {
                lang: {
                    category: [t[1:] for t in pairs]
                    for category, pairs in categories.items()
                }
                for lang, categories in attributes.items()
            }
            
        json.dump(to_save[lang], f, indent=4, ensure_ascii=False)


es_AR
19 [(2, 'un', 'una'), (9, 'los', 'las'), (15, 'mujeres', 'hombres'), (38, 'Lo', 'La'), (44, 'mujeres.', 'hombres.'), (70, 'el', 'la'), (72, 'chica', 'chico'), (83, 'ella', 'él'), (116, 'buenos', 'buenas'), (153, 'trans.', 'cis.'), (158, 'al', 'a'), (158, 'hombre', 'la'), (174, 'Juan', 'María'), (193, 'Tomás', 'Juana'), (270, 'mujer,', 'hombre,'), (288, 'chicos', 'chicas'), (292, 'mujer', 'hombre'), (389, 'mujer.', 'hombre.'), (610, 'Marcos', 'María')]
POS 13
NER: 0
6 [(15, 'mujeres', 'hombres'), (72, 'chica', 'chico'), (83, 'ella', 'él'), (116, 'buenos', 'buenas'), (288, 'chicos', 'chicas'), (292, 'mujer', 'hombre')]
22 [(4, 'asiáticos', 'occidentales'), (7, 'negro.', 'cheto.'), (8, 'negros', 'chetos'), (65, 'boliviano.', 'europeo.'), (66, 'mapuches', 'porteños'), (68, 'negros.', 'chetos.'), (95, 'negro', 'cheto'), (123, 'de', 'del'), (123, 'la', 'country.'), (138, 'pobres', 'ricos'), (243, 'bolivianos', 'europeos'), (295, 'chinos', 'europeos'), (303, 'Quimey', 'Guillermo'), (331